# 02 Silver: Data Cleaning
- Removes duplicate rows and rows with missing income
- Fixes the date format and removes unrealistic ages
- Cleans marital status (Absurd and YOLO removed; groups: Partnered and Single)
- Removes the income outlier (above 200,000)
- Adds new columns: Age, AgeGroup, TotalSpend, TotalKids, IncomeGroup, JoinYear
- **Output table:** marketing.default.silver_campaign (2,208 rows)

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("marketing.default.bronze_campaign")

silver = (bronze
    # 1. remove duplicate rows
    .dropDuplicates()
    # 2. remove rows with missing income
    .dropna(subset=["Income"])
    # 3. fix the date column
    .withColumn("Dt_Customer", F.to_date("Dt_Customer", "dd-MM-yyyy"))
    # 4. create Age
    .withColumn("Age", F.year(F.current_date()) - F.col("Year_Birth"))
    # 5. total spending
    .withColumn("TotalSpend",
        F.col("MntWines") + F.col("MntFruits") + F.col("MntMeatProducts")
        + F.col("MntFishProducts") + F.col("MntSweetProducts") + F.col("MntGoldProds"))
    # 6. total children
    .withColumn("TotalKids", F.col("Kidhome") + F.col("Teenhome"))
    # 7. remove unrealistic ages (outliers)
    .filter(F.col("Age") < 100)
    # 8. group ages
    .withColumn("AgeGroup",
        F.when(F.col("Age") < 30, "Under 30")
         .when(F.col("Age") < 45, "30-44")
         .when(F.col("Age") < 60, "45-59")
         .otherwise("60+"))
        # clean marital status
    .withColumn("Marital_Status",
        F.when(F.col("Marital_Status").isin("Married", "Together"), "Partnered")
         .when(F.col("Marital_Status").isin("Single", "Alone", "Divorced", "Widow"), "Single")
         .otherwise(None))
    .filter(F.col("Marital_Status").isNotNull())      # removes Absurd, YOLO
    # remove income outlier
    .filter(F.col("Income") < 200000)
    # income groups
    .withColumn("IncomeGroup",
        F.when(F.col("Income") < 30000, "Under 30K")
         .when(F.col("Income") < 60000, "30K-60K")
         .when(F.col("Income") < 90000, "60K-90K")
         .otherwise("90K+"))
    .withColumn("JoinYear", F.year("Dt_Customer"))
)

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("marketing.default.silver_campaign")
display(silver.limit(10))

ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Age,TotalSpend,TotalKids,AgeGroup,IncomeGroup,JoinYear
5524,1957,Graduation,Single,58138,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1,69,1617,0,60+,30K-60K,2012
2174,1954,Graduation,Single,46344,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0,72,27,2,60+,30K-60K,2014
4141,1965,Graduation,Partnered,71613,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0,61,776,0,60+,60K-90K,2013
6182,1984,Graduation,Partnered,26646,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0,42,53,1,30-44,Under 30K,2014
5324,1981,PhD,Partnered,58293,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0,45,422,1,45-59,30K-60K,2014
7446,1967,Master,Partnered,62513,0,1,2013-09-09,16,520,42,98,0,42,14,2,6,4,10,6,0,0,0,0,0,0,3,11,0,59,716,1,45-59,60K-90K,2013
965,1971,Graduation,Single,55635,0,1,2012-11-13,34,235,65,164,50,49,27,4,7,3,7,6,0,0,0,0,0,0,3,11,0,55,590,1,45-59,30K-60K,2012
6177,1985,PhD,Partnered,33454,1,0,2013-05-08,32,76,10,56,3,1,23,2,4,0,4,8,0,0,0,0,0,0,3,11,0,41,169,1,30-44,30K-60K,2013
4855,1974,PhD,Partnered,30351,1,0,2013-06-06,19,14,0,24,3,3,2,1,3,0,2,9,0,0,0,0,0,0,3,11,1,52,46,1,45-59,30K-60K,2013
5899,1950,PhD,Partnered,5648,1,1,2014-03-13,68,28,0,6,1,1,13,1,1,0,0,20,1,0,0,0,0,0,3,11,0,76,49,2,60+,Under 30K,2014
